In [ ]:
import os
import re
import glob
import time
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt

from matplotlib.ticker import AutoMinorLocator
# %matplotlib tk
from JunoPy.index import IndexFile
from JunoPy.mag import MagData, mva
from JunoPy.waves import WavesDataDas
from JunoPy.jedi import JediData
from JunoPy.library import find_ticks_between

In [ ]:
# startevent = dt.datetime.strptime('2018-085T11:35:20', '%Y-%jT%H:%M:%S')
# endevent   = dt.datetime.strptime('2018-085T11:35:44', '%Y-%jT%H:%M:%S')
# startevent = dt.datetime.strptime('2017-236T20:21:15', '%Y-%jT%H:%M:%S')
# endevent   = dt.datetime.strptime('2017-236T20:21:38', '%Y-%jT%H:%M:%S')
startevent = dt.datetime.strptime('2017-338T01:49:57', '%Y-%jT%H:%M:%S')
endevent   = dt.datetime.strptime('2017-338T01:51:00', '%Y-%jT%H:%M:%S')
# startevent = dt.datetime.strptime('2017-338T10:38:57', '%Y-%jT%H:%M:%S')
# endevent   = dt.datetime.strptime('2017-338T11:08:00', '%Y-%jT%H:%M:%S')
start = startevent - dt.timedelta(minutes=100)
end   = endevent + dt.timedelta(minutes=100)

In [ ]:
magfile = glob.glob('data/mag/*{:s}ss*.hdf5'.format(start.strftime('%Y%j')))[0]
magdata = MagData(filename=magfile)

wavesdata = WavesDataDas(start, end)

## Plot data in JSS coordinate system

In [ ]:
def plot_data(magdata, wavesdata, start, end, startevent=None, endevent=None, outfolder=None):
    """Plot data between start and end."""
    
    # Slice the data.
    magdata.df = magdata.df.loc[start:end]
    
    # Find the ticks.
    timearray, tickstring = find_ticks_between(start, end)
    
    # Create the figure.
    fig, axs = plt.subplots(5, 1, 
                            gridspec_kw={'hspace': 0, 'right':0.9, 'top':0.9}, 
                            figsize=(9, 6), 
                            dpi=300)
    
    axs[:4] = magdata.plot(vars=['Br', 'Btheta', 'Bphi', 'Bmag'], axs=axs[:4])
    axs[4] = wavesdata.plot(axs[4], freqlimit=True, vmin=1e-14, vmax=1e-10, )
    
        
    for a in axs:
        # a.set_xlim([timearray[0], timearray[-1]])
        a.set_xticks(timearray)
        a.set_xticklabels(tickstring)
        a.label_outer()
        # a.axvline(startevent, color='red', lw=1.)
        # a.axvline(endevent, color='red', lw=1.)
        
            
    timesp = [
        '2017-338T01:41:00 2017-338T01:44:00',
        '2017-338T01:30:30 2017-338T01:34:00',
        '2017-338T00:57:41 2017-338T01:03:40',
        '2017-338T01:51:00 2017-338T01:52:30'
    ]
    
    timesn = [
        '2017-338T01:33:55 2017-338T01:41:00',
        '2017-338T01:22:15 2017-338T01:23:35',
        '2017-338T00:52:41 2017-338T00:54:50'
        
        
    ]
    
#     for a in axs[:-1]:
#         a.axvspan(startevent, endevent,
#                   facecolor=[0.7, 0.2, 0.2], alpha=0.3)
        
#         for t in timesp:
#             a.axvspan(dt.datetime.strptime(t.split()[0], '%Y-%jT%H:%M:%S'), 
#                       dt.datetime.strptime(t.split()[1], '%Y-%jT%H:%M:%S'),
#                       facecolor=[0.2, 0.2, 0.7], alpha=0.2)
            
#         for t in timesn:
#             a.axvspan(dt.datetime.strptime(t.split()[0], '%Y-%jT%H:%M:%S'), 
#                       dt.datetime.strptime(t.split()[1], '%Y-%jT%H:%M:%S'),
#                       facecolor=[0.7, 0.2, 0.2], alpha=0.2)
    
    axs[:4] = magdata.plot_between(startevent,
                                   endevent,
                                   vars=['Br', 'Btheta', 'Bphi', 'Bmag'], 
                                   axs=axs[:4],
                                   color='b')
    for t in timesp:
        axs[:4] = magdata.plot_between(dt.datetime.strptime(t.split()[0], '%Y-%jT%H:%M:%S'),
                                       dt.datetime.strptime(t.split()[1], '%Y-%jT%H:%M:%S'),
                                       vars=['Br', 'Btheta', 'Bphi', 'Bmag'], 
                                       axs=axs[:4],
                                       color='r')
        
    for t in timesn:
        axs[:4] = magdata.plot_between(dt.datetime.strptime(t.split()[0], '%Y-%jT%H:%M:%S'),
                                       dt.datetime.strptime(t.split()[1], '%Y-%jT%H:%M:%S'),
                                       vars=['Br', 'Btheta', 'Bphi', 'Bmag'], 
                                       axs=axs[:4],
                                       color='b')
            
    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)
    
    axs[1].annotate('Event #2', xy=(startevent + dt.timedelta(seconds=120), -2),
                    color='b',
                    arrowprops=dict(facecolor='red', shrink=0.005),
                    horizontalalignment='left', verticalalignment='top')

    
    plt.suptitle(dt.datetime.strftime(start, '%Y-%m-%d') + \
                 ', DOY={:s}'.format(dt.datetime.strftime(start, '%j')), y=0.94)
    
    if outfolder:
        fig.savefig(outfolder + dt.datetime.strftime(start, '%Y-%jT%H:%M') + '.png')
        plt.close(fig)
    
    # fig.savefig('out_images/Figure4_try1.png')
    
    # plt.close(fig)
    
    
plot_data(magdata, wavesdata, start, end, startevent=startevent, endevent=endevent, outfolder='out_images/')

## Plot data in MVA coordinate system

In [ ]:
def plot_data_mva(magdata, wavesdata, start, end, startevent=None, endevent=None, outfolder=None):
    """Plot data between start and end."""
    
    # Slice the data.
    magdata.df = magdata.df.loc[start:end]
    magdata = mva(magdata)
    
    # Find the ticks.
    timearray, tickstring = find_ticks_between(start, end)
    
    # Create the figure.
    fig, axs = plt.subplots(5, 1, 
                            gridspec_kw={'hspace': 0, 'right':0.9, 'top':0.9}, 
                            figsize=(9, 6), 
                            dpi=300)
    
    axs[:4] = magdata.plot(vars=['B1', 'B2', 'B3', 'Bmag'], axs=axs[:4])
    axs[4] = wavesdata.plot(axs[4], freqlimit=True, vmin=1e-14, vmax=1e-10, )
    
        
    for a in axs:
        # a.set_xlim([timearray[0], timearray[-1]])
        a.set_xticks(timearray)
        a.set_xticklabels(tickstring)
        a.label_outer()
        # a.axvline(startevent, color='red', lw=1.)
        # a.axvline(endevent, color='red', lw=1.)
        
            
    timesp = [
        '2017-338T01:41:00 2017-338T01:44:00',
        '2017-338T01:30:30 2017-338T01:34:00',
        '2017-338T00:57:41 2017-338T01:03:40'
    ]
    
    timesn = [
        '2017-338T01:33:55 2017-338T01:41:00',
        '2017-338T01:22:15 2017-338T01:23:35'
        
        
    ]
    
    for a in axs[:-1]:
        a.axvspan(startevent, endevent,
                  facecolor=[0.7, 0.2, 0.2], alpha=0.3)
        
        for t in timesp:
            a.axvspan(dt.datetime.strptime(t.split()[0], '%Y-%jT%H:%M:%S'), 
                      dt.datetime.strptime(t.split()[1], '%Y-%jT%H:%M:%S'),
                      facecolor=[0.2, 0.2, 0.7], alpha=0.2)
            
        for t in timesn:
            a.axvspan(dt.datetime.strptime(t.split()[0], '%Y-%jT%H:%M:%S'), 
                      dt.datetime.strptime(t.split()[1], '%Y-%jT%H:%M:%S'),
                      facecolor=[0.7, 0.2, 0.2], alpha=0.2)
            
    a.tick_params(axis='x', width=2, rotation=0)
    a.xaxis.label.set_visible(False)
    
    axs[1].annotate('Event #2', xy=(0.46, 0.3),  xycoords='axes fraction',
                    xytext=(0.55, 0.3), textcoords='axes fraction',
                    arrowprops=dict(facecolor='red', shrink=0.005),
                    horizontalalignment='left', verticalalignment='top')

    
    plt.suptitle(dt.datetime.strftime(start, '%Y-%m-%d') + \
                 ', DOY={:s}'.format(dt.datetime.strftime(start, '%j')), y=0.94)
    
    if outfolder:
        fig.savefig(outfolder + dt.datetime.strftime(start, '%Y-%jT%H:%M') + '.png', transparent=False)
        plt.close(fig)
    
    fig.savefig('out_images/Figure4_try1_MVA.png')
    
    # plt.close(fig)
    
    
plot_data_mva(magdata, wavesdata, start, end, startevent=startevent, endevent=endevent)